# 🏋️ Module 3.5: Exercises

Practice experiment management and search/filter operations.

---

In [ ]:
import mlflow
from mlflow import MlflowClient
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.autolog(disable=True)
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Ready!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Exercise 1: Create Organized Experiments

Create two experiments:
1. `"ex3_baseline"` with tags: `phase=baseline`, `team=ml`
2. `"ex3_optimization"` with tags: `phase=optimization`, `team=ml`

Add 2 runs to each experiment with different model types.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
# Create experiments (handle case where they already exist)
for name, phase in [("ex3_baseline", "baseline"), ("ex3_optimization", "optimization")]:
    try:
        mlflow.create_experiment(name, tags={"phase": phase, "team": "ml"})
    except mlflow.exceptions.MlflowException:
        pass  # Already exists

# Add runs to baseline
mlflow.set_experiment("ex3_baseline")
for name, model in [("lr_base", LogisticRegression(max_iter=1000, random_state=42)),
                     ("rf_base", RandomForestClassifier(n_estimators=50, random_state=42))]:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))

# Add runs to optimization
mlflow.set_experiment("ex3_optimization")
for name, model in [("rf_tuned", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)),
                     ("gb_tuned", GradientBoostingClassifier(n_estimators=200, random_state=42))]:
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        mlflow.log_metric("accuracy", accuracy_score(y_test, model.predict(X_test)))

print("✅ Experiments created with runs!")

## Exercise 2: Search Across Experiments

Find the best run **across both experiments** (`ex3_baseline` and `ex3_optimization`) by accuracy. Print the run name, experiment, and accuracy.

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
all_runs = mlflow.search_runs(
    experiment_names=["ex3_baseline", "ex3_optimization"],
    order_by=["metrics.accuracy DESC"]
)

best = all_runs.iloc[0]
print(f"🏆 Best overall:")
print(f"   Run: {best.get('tags.mlflow.runName', 'N/A')}")
print(f"   Experiment ID: {best['experiment_id']}")
print(f"   Accuracy: {best['metrics.accuracy']:.4f}")

## Exercise 3: Build a Run Comparison Report

Write a function `compare_runs(experiment_name)` that:
1. Gets all runs from the experiment
2. Returns a clean DataFrame with: run name, model type (if available), accuracy, f1
3. Sorted by accuracy descending
4. Prints a summary

In [ ]:
# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION
def compare_runs(experiment_name):
    runs = mlflow.search_runs(
        experiment_names=[experiment_name],
        order_by=["metrics.accuracy DESC"]
    )
    
    if len(runs) == 0:
        print(f"No runs found in '{experiment_name}'")
        return pd.DataFrame()
    
    # Build clean report
    report_data = []
    for _, run in runs.iterrows():
        report_data.append({
            "Run Name": run.get("tags.mlflow.runName", "unnamed"),
            "Accuracy": run.get("metrics.accuracy", None),
            "F1": run.get("metrics.f1", None),
            "Status": run.get("status", "unknown"),
        })
    
    report = pd.DataFrame(report_data)
    
    print(f"📊 Experiment: {experiment_name}")
    print(f"   Total runs: {len(report)}")
    if report['Accuracy'].notna().any():
        print(f"   Best accuracy: {report['Accuracy'].max():.4f}")
        print(f"   Avg accuracy: {report['Accuracy'].mean():.4f}")
    print()
    
    return report

report = compare_runs("ex3_baseline")
report

---

## 🎓 Module 3 Complete!

You've mastered:
- ✅ Creating and managing experiments with tags
- ✅ Searching and filtering runs with powerful query syntax
- ✅ Using the MLFlow UI for visual analysis
- ✅ Building reusable functions for run comparison

**Next up**: Module 4 — Model Registry →